In [ ]:
import numpy as np
import torch
import torch.nn as nn

from dlfs_autograd.layers import RecurrentLayer
from dlfs_autograd.modules import RNN
from dlfs_autograd.tensor.base import Parameter, Tensor
from .helpers import compare_arrays

In [ ]:
use_autograd = True

np.random.seed(0)
torch.manual_seed(0)

B = 120
T = 25
C = 24
hidden_size = 32
n_layers = 3

x = Tensor(
    np.random.normal(size=(B, T, C))
)
print(x.shape)

In [ ]:
torch_rnn = nn.RNN(
    input_size=C,
    hidden_size=hidden_size,
    num_layers=n_layers,
    batch_first=True
)

with torch.no_grad():
    for name, param in torch_rnn.named_parameters():
        if str(name).find('bias') != -1:
            param.zero_() # Set all bias params to zero

In [ ]:
torch_x = torch.Tensor(x.data)
h0 = torch.zeros((n_layers, B, hidden_size))

torch_out, _ = torch_rnn(torch_x, h0)
print(f'Torch RNN out: {torch_out.shape}')

In [ ]:
my_rnn = RNN(
    n_inputs=C, 
    n_hidden=hidden_size, 
    n_layers=n_layers, 
    predict_sequence=True
)

for idx in range(n_layers):
    my_rnn.recurrent_layers[idx].input_bias = Parameter(
        np.zeros(hidden_size)
    )

for name, param in torch_rnn.named_parameters():
    naziv, vrsta, index = str(name).split('_')
    index = int(index[1:])
    if naziv == "weight":
        if vrsta == "ih":
            i_weights = Parameter(
                param.detach().numpy().T
            )
            print(f'Setting {naziv}_{vrsta}_{index} to recurrent layer {index}')
            my_rnn.recurrent_layers[index].input_weights = i_weights
        elif vrsta == "hh":
            h_weights = Parameter(
                param.detach().numpy().T
            )
            print(f'Setting {naziv}_{vrsta}_{index} to recurrent layer {index}')
            my_rnn.recurrent_layers[index].hidden_weights = h_weights

In [ ]:
my_rnn.forward(x)
dlfs_out = my_rnn.output
print(dlfs_out.shape)

In [ ]:
compare_arrays(torch_out.detach().numpy(), dlfs_out.data)

In [ ]:
torch_sum = torch.sum(torch_out)
print(torch_sum)
torch_sum.backward()

In [ ]:
dlfs_sum = dlfs_out.sum()
print(dlfs_sum)
dlfs_sum.backward()

In [ ]:
for name, param in torch_rnn.named_parameters():
    
    naziv, vrsta, index = str(name).split('_')
    index = int(index[1:])

    if naziv == "weight":
        if vrsta == "ih":
            torch_grad = param.grad.detach().numpy().T
            dlfs_grad = my_rnn.recurrent_layers[index].input_weights.grad
            print(f'---Comparing layer {index} input weights grads---')
            compare_arrays(torch_grad, dlfs_grad, 1e-5, 1e-5)
        elif vrsta == "hh":
            torch_grad = param.grad.detach().numpy().T
            dlfs_grad = my_rnn.recurrent_layers[index].hidden_weights.grad
            print(f'---Comparing layer {index} hidden weights grads---')
            compare_arrays(torch_grad, dlfs_grad, 1e-5, 1e-5)